In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import time

nb_dir = Path(os.getcwd()).resolve()
repo_root = nb_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams, ensure_bem_field
from fracture_utils.Ubem.disk_network_propagation import CrackGrowthParams, run_network_growth_uncoupled
from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook, plot_total_field
from fracture_utils.Ubem.coupling_blocks_v2 import BEMBlockV2, compose_solver_kwargs_v2
from fracture_utils.Ubem.disk_iterative_coupling import (
    compute_crack_boundary_tractions_direct,
    solve_bem_with_extra_boundary_tractions,
)
from fracture_utils.Ubem.boundary_conditions import build_boundary

# -----------------------
# Output + mode selection
# -----------------------
bem_dir = repo_root / "output" / "BEM_Brazilian_disk_2"
bem_dir.mkdir(parents=True, exist_ok=True)

# Choose one:
#  - "no_coupling"
#  - "iterative"  (outer-cycle updates only)
#  - "direct"     (outer-cycle updates only)
COUPLING_METHOD = "no_coupling"
OUTER_CYCLES = 3

# -----------------------
# Network (YOU control)
# -----------------------
mm = 1e-3
vertices = np.array([
    [0, -2.0*mm, -3.0*mm],
    [1,  2.0*mm,  3.0*mm],
], float)
connectivity = np.array([[0, 1]], int)

# -----------------------
# (1) BEM baseline
# -----------------------
SKIP_BEM_SOLVE = True
disk = BrazilianDiskParams(R=25.4e-3/2, P_total=3.8e3, E=231.52e9, nu=0.3, n_elem=120, n_grid=120)

bem_block = BEMBlockV2(disk=disk, out_dir=bem_dir)
ensure_bem_field(
    disk,
    bem_dir,
    recompute=(not SKIP_BEM_SOLVE),
    show=True,
    save_contours=True,
)

boundary_mesh = build_boundary({"type": "circle", "R": disk.R, "n_boundary": disk.n_elem, "center": (0.0, 0.0)})

# -----------------------
# Plot options
# -----------------------
plot_params = PlotParams(
    cmap="jet",
    match_limits_to_crack=True,
    robust=True,
    robust_pct=97.0,
    symmetric=True,
    vmax_factor=2.0,
    n_levels=60,
    n_line_levels=15,
    dpi=300,
)

hook = make_standard_plot_hook(
    bem_dir=bem_dir,
    combined_out_dir=bem_dir,
    plot_params=plot_params,
    show_initial=True,
    show_cycles=True,
    show_final=True,
)


def _network_to_arrays(net):
    V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], float)
    C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], int) if len(net.edges) else np.zeros((0,2), int)
    return V, C


def _save_total_as_bem_arrays(Sxx_tot, Syy_tot, Sxy_tot):
    xs = np.load(bem_dir / "xs.npy")
    ys = np.load(bem_dir / "ys.npy")
    np.save(bem_dir / "xs.npy", xs)
    np.save(bem_dir / "ys.npy", ys)
    np.save(bem_dir / "Sxx.npy", np.asarray(Sxx_tot, float))
    np.save(bem_dir / "Syy.npy", np.asarray(Syy_tot, float))
    np.save(bem_dir / "Sxy.npy", np.asarray(Sxy_tot, float))


# One-way growth kwargs (used for propagation increments)
one_way_kwargs = compose_solver_kwargs_v2("one_way", base_solver_kwargs=None)

# Direct kwargs (used only at outer-cycle coupling checkpoints)
aug_payload = bem_block.build_augmented_payload(d_mode="correction_zero", gauss_n=4)
direct_kwargs = compose_solver_kwargs_v2(
    "full_kkt",
    base_solver_kwargs={
        "augmented_enforce_crack_equilibrium": True,
        "augmented_equilibrate": True,
        "augmented_equilibrate_cols": False,
        "augmented_physical_scaling": True,
        "augmented_scale_traction_rows": True,
    },
    augmented_payload=aug_payload,
    augmented_ridge_q=1e-4,
    augmented_ridge_y=1e-8,
    augmented_keep_bem_applied=True,
)

V_curr = vertices.copy()
C_curr = connectivity.copy()
res_last = None

t_start_total = time.perf_counter()
t_kkt_total = 0.0

for cyc in range(1, int(OUTER_CYCLES) + 1):
    print(f"\n[outer-cycle] {cyc}/{OUTER_CYCLES} method={COUPLING_METHOD}")

    # Growth for ONE outer cycle using one-way applied field from current bem_dir arrays
    grow = CrackGrowthParams(max_cycles=1, max_increments_per_cycle=5, L_limit_mm=20.0, vertex_high=18, solver_kwargs=one_way_kwargs)
    out_growth = bem_dir / f"outer_{cyc:02d}_growth"
    out_growth.mkdir(parents=True, exist_ok=True)

    res_growth = run_network_growth_uncoupled(
        bem_dir=bem_dir,
        out_dir=out_growth,
        vertices=V_curr,
        connectivity=C_curr,
        params=grow,
        plot_hook=hook,
    )

    # Outer-cycle coupling update (only here)
    if COUPLING_METHOD == "iterative":
        tx_cr, ty_cr = compute_crack_boundary_tractions_direct(res=res_growth, boundary_mesh=boundary_mesh)
        solve_bem_with_extra_boundary_tractions(
            disk_params=disk,
            bem_dir=bem_dir,
            tx_extra=-tx_cr,
            ty_extra=-ty_cr,
            show=False,
        )
        res_last = res_growth

    elif COUPLING_METHOD == "direct":
        # Solve direct KKT on current network (no additional growth), then update bem_dir arrays
        grow_direct = CrackGrowthParams(max_cycles=0, L_limit_mm=20.0, vertex_high=18, solver_kwargs=direct_kwargs)
        out_direct = bem_dir / f"outer_{cyc:02d}_direct"
        out_direct.mkdir(parents=True, exist_ok=True)

        Vg, Cg = _network_to_arrays(res_growth.calc.network)
        t_kkt0 = time.perf_counter()
        res_direct = run_network_growth_uncoupled(
            bem_dir=bem_dir,
            out_dir=out_direct,
            vertices=Vg,
            connectivity=Cg,
            params=grow_direct,
            plot_hook=None,
        )
        t_kkt_total += time.perf_counter() - t_kkt0

        Sxx_tot, Syy_tot, Sxy_tot = plot_total_field(
            bem_dir=bem_dir,
            res=res_direct,
            out_dir=out_direct,
            tag=f"outer_{cyc:02d}_direct",
            params=plot_params,
            show=True,
        )
        _save_total_as_bem_arrays(Sxx_tot, Syy_tot, Sxy_tot)
        res_last = res_direct

    else:
        # no_coupling
        res_last = res_growth

    # advance network for next outer cycle from growth result
    V_curr, C_curr = _network_to_arrays(res_growth.calc.network)

t_total = time.perf_counter() - t_start_total
print(f"\n[done] outer-cycle run completed")
print(f"Total run time: {t_total:.2f} s")
print(f"BEM-KKT solver time: {t_kkt_total:.2f} s")
